In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import expm, expm_multiply
from numpy.linalg import eigh
from tqdm import tqdm
import matplotlib.pyplot as plt

from sympy import Matrix
import cvxpy as cp
import random

import h5py

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],   # default LaTeX font
    "text.latex.preamble": r"\usepackage{amsmath}",
    "font.size": 15, 
})

In [2]:
# PARAMETERS

# Lattice geometry
X = 32
Y = 34

# Number of edges
Ex = Y*(X-1)
Ey = X*(Y-1)
R = Ex + Ey

# Number of modes
M = X*Y
M_occ = (M-X)//2

# Hamiltonian engineering
N = 3**M # Total number of pulses
ratio = 6 # Column to row ratio
K = ratio*R # Number of pulses (columns) to be considered

In [3]:
# FUNCTIONS

def site_to_index(x,y):
    if x >= X or y >= Y:
        print('x needs to be less than ', X, ' and, y needs to be less than', Y)
    else:
        return x + X*(y)

def index_to_site(m):
    x = m % X
    y = m // X
    return (x,y)


def square_lattice_edge_vertex_incidence(Lx: int, Ly: int, *, periodic: bool = False) -> np.ndarray:
    """
    Build an (E x V) array M for an Lx-by-Ly square lattice.

    Columns = vertices, indexed by v = x + Lx*y.
    Rows    = undirected edges (each nearest-neighbor bond listed once).

    For each edge-row e = (u, v):
        M[e, u] = 1
        M[e, v] = 2
        all other entries 0.

    If periodic=False: open boundary conditions.
    If periodic=True : wrap-around in both x and y directions.
    """
    if Lx <= 0 or Ly <= 0:
        raise ValueError("Lx and Ly must be positive integers.")

    def vid(x: int, y: int) -> int:
        return x + Lx * y

    edges = []

    # Horizontal edges: (x,y) -- (x+1,y)
    for y in range(Ly):
        for x in range(Lx):
            x2 = (x + 1) % Lx
            if (not periodic) and (x + 1 >= Lx):
                continue
            u = vid(x, y)
            v = vid(x2, y)
            edges.append((u, v))

    # Vertical edges: (x,y) -- (x,y+1)
    for y in range(Ly):
        for x in range(Lx):
            y2 = (y + 1) % Ly
            if (not periodic) and (y + 1 >= Ly):
                continue
            u = vid(x, y)
            v = vid(x, y2)
            edges.append((u, v))

    V = Lx * Ly
    E = len(edges)
    M = np.zeros((E, V), dtype=np.int8)

    # Put 1 on the first endpoint, 2 on the second endpoint
    for e, (u, v) in enumerate(edges):
        if u == v:
            raise RuntimeError("Self-edge encountered (should not happen for square lattice).")
        M[e, u] = 1
        M[e, v] = 2

    return M

#------------------------------------------------------------------------------------------------
# Function which takes a number between 0 and N-1 and converts it to a base 3 array

def tritarray(x,M):
    tritarray = np.zeros(shape = M)
    for j in range(M):
        r = x % 3
        x = x // 3 
        tritarray[-(j+1)] = r
    return tritarray

#------------------------------------------------------------------------------------------------
# Function for selecting K random pulses on M modes

def random_pulses(M,K):
    
    N = 3**M
    
    sample = random.sample(range(N),K) 
    pulses = np.zeros(shape = (K,M)) # K pulses of length M
    
    for j in range(K):
        pulses[j] = tritarray(sample[j],M)
        
    return pulses
#------------------------------------------------------------------------------------------------
# Function for selecting K random pulses on M modes efficiently (with replacement)

def random_pulses_eff(M, K, rng=None):
    """K uniformly random tritstrings of length M, entries in {0, 1, 2}."""
    if rng is None:
        rng = np.random.default_rng()
    return rng.integers(0, 3, size=(K, M), dtype=np.int8)
#------------------------------------------------------------------------------------------------

# Check if F is feasible
def check_feasible(F):

    x = cp.Variable(F.shape[1])
    objective = cp.Minimize(cp.sum(x))
    constraints = [F @ x == 0, x >= 1]

    # Define and solve the problem
    problem = cp.Problem(objective, constraints)
    problem.solve(solver = cp.MOSEK)

    # # Print solution
    # print("Optimal value:", problem.value)
    # print("Optimal x:", x.value)
    
    LP_has_solution = 0
    F_has_full_rank = 0
    
    if type(x.value) == np.ndarray:
        LP_has_solution = 1
    if F.shape[0]- np.linalg.matrix_rank(F)==0:
        F_has_full_rank = 1
    feasible = LP_has_solution * F_has_full_rank
    
    return bool(feasible)

#------------------------------------------------------------------------------------------------
# Function for finding the constraint matrix with given pulses

def constraint_matrix_given_pulses(R,K,rows,pulses):

    F = np.zeros(shape = (R,K), dtype=np.complex128)
    w = np.exp(1j*2*np.pi/3)

    for i in range(R):
        for j in range(K):
            power = rows[i]@pulses[j]
            F[i][j] = w**power

    return F
#------------------------------------------------------------------------------------------------
# Hamiltonian Engineering
def hamiltonian_engineering(R,K,rows,pulses,beta):

    F = constraint_matrix_given_pulses(R,K,rows,pulses)
    
    # print(check_feasible(F))

    x = cp.Variable(K)
    objective = cp.Minimize(cp.sum(x))
    constraints = [F @ x == beta, x>=0]

    # Define and solve the problem
    problem = cp.Problem(objective, constraints)
    problem.solve(solver = cp.MOSEK)

    # # Print solution
    # print("Optimal value:", problem.value)
    # print("Optimal x:", x.value)
    
    return problem.value, x.value, check_feasible(F)

#------------------------------------------------------------------------------------------------
# Function for adding the conjugate components to the coefficient vector
def conjugate_components(vec,number_x_edges,number_y_edges):
    number_edges = number_x_edges + number_y_edges
    vec2 = np.zeros(2*number_edges, dtype=complex)
    for i in range(number_x_edges):
        vec2[i] = vec[i]
        vec2[i+number_x_edges] = vec[i].conj()
    for j  in range (number_y_edges):
        vec2[j + 2*number_x_edges] = vec[j+number_x_edges]
        vec2[j+2*number_x_edges+number_y_edges] = vec[j+number_x_edges].conj()
        
    return vec2

#------------------------------------------------------------------------------------------------
def edge_indices(Lx=X,Ly=X):

    num_x_edges = Y*(X-1)
    num_y_edges = X*(Y-1)
    num_edges = num_x_edges + num_y_edges
    
    row_ind = np.zeros(2*num_edges, dtype=int)
    col_ind = np.zeros(2*num_edges, dtype=int)

    e=0
    
    # X edges
    for y in range(Y):    
        for x in range(X-1):
            j = site_to_index(x,y)
            k = site_to_index(x+1,y)
            row_ind[e] = j
            col_ind[e] = k
            e += 1
            
    # X edges transposed
    for y in range(Y):    
        for x in range(X-1):
            j = site_to_index(x,y)
            k = site_to_index(x+1,y)
            row_ind[e] = k
            col_ind[e] = j
            e += 1
            
    # Y edges
    for y in range(Y-1):    
        for x in range(X):
            j = site_to_index(x,y)
            k = site_to_index(x,y+1)
            row_ind[e] = j
            col_ind[e] = k
            e += 1
    # Y edges transposed
    for y in range(Y-1):    
        for x in range(X):
            j = site_to_index(x,y)
            k = site_to_index(x,y+1)
            row_ind[e] = k
            col_ind[e] = j
            e += 1
            
    return row_ind,col_ind

#------------------------------------------------------------------------------------------------
# Function for finding the ground state of a quadratic Hamiltonian (written by Claude, understand better)
from scipy.sparse.linalg import eigsh

def ground_state_rdm(h, M, force_dense=False):
    """
    1-RDM of the M-particle ground state of H = sum_{ij} h_{ij} c_i^† c_j.

    Convention: rho_{ij} = <c_j^† c_i> = (Phi Phi^†)_{ij}.

    Parameters
    ----------
    h : (N, N) Hermitian, dense or sparse
    M : int, particle number
    force_dense : if True, always use np.linalg.eigh

    Returns
    -------
    Phi  : (N, M) complex, columns = occupied orbitals (orthonormal)
    rho  : (N, N) complex Hermitian, rank-M projector
    eps  : (M,) sorted single-particle energies of occupied orbitals
    E_gs : float, ground-state energy = sum(eps)
    """
    N = h.shape[0]
    use_sparse = sp.issparse(h) and (M < (N+2) // 2) and not force_dense
    if use_sparse:
        print('Using sparse')
        eps, Phi = eigsh(h, k=M, which='SA')
        order = np.argsort(eps)
        eps, Phi = eps[order], Phi[:, order]
    else:
        h_dense = h.toarray() if sp.issparse(h) else h
        all_eps, U = np.linalg.eigh(h_dense)
        eps = all_eps[:M]
        Phi = U[:, :M]
    rho  = np.array(Phi @ Phi.conj().T)
    E_gs = float(eps.sum())
    return Phi, rho, eps, E_gs

#------------------------------------------------------------------------------------------------
#  Function for extracting the nonzero pulses and evolution times from the output of the LP
def nonzero_pulses(pulses,times, M=M):
    
    rounded_times = np.round(times,5)
    nonzero_times = rounded_times[ rounded_times != 0]
    nonzero_indices = np.nonzero(rounded_times)[0]
    nonzero_pulses = np.zeros((len(nonzero_indices),M))
    for k in range(len(nonzero_indices)):
        nonzero_pulses[k] = pulses[nonzero_indices[k]]
    return nonzero_pulses,nonzero_times

In [4]:
row_ind,col_ind = edge_indices(X,Y)
matrix_ind = (row_ind,col_ind)

In [5]:
# System Hamiltonian coefficients

# Homogeneous tunnelling coefficients
t = 1

# coefficient vector with conjugate termsc
alpha2 = np.zeros(2*R)

for e in range(2*R):
    alpha2[e] = t
    
# System Hamiltonian

h_system_csr = sp.csr_matrix((alpha2,(row_ind,col_ind)))
h_system_csc = h_system_csr.tocsc()

(h_system_csr - h_system_csr.conj().T).nnz == 0

True

In [6]:
# Target Hamiltonian (Harper-Hofstadter)

flux = 1/3
phase = np.exp(1j*2*np.pi*flux)

# Target Hamiltonian coefficients
beta_HH = np.zeros(R,dtype=complex)
e = 0

# X edges
for y in range(Y):
    for x in range(X-1):
        m = y % 3
        beta_HH[e] = phase**(m)
        e += 1

# Y edges
for y in range(Y-1):
    for x in range(X):
        beta_HH[e] = 1
        e += 1
beta_HH2 = conjugate_components(beta_HH,Ex,Ey)

h_HH_csr = sp.csr_matrix((beta_HH2,(row_ind,col_ind)))
h_HH_csc = sp.csc_matrix((beta_HH2,(row_ind,col_ind)))

(h_HH_csr - h_HH_csr.conj().T).nnz == 0

True

In [7]:

rows = square_lattice_edge_vertex_incidence(X,Y)

In [ ]:
# # Hamiltonian engineering

# rows = square_lattice_edge_vertex_incidence(X,Y)
# pulses = random_pulses_eff(M,K)
# qtime,times,surjective = hamiltonian_engineering(R,K,rows,pulses,beta_HH)

# nonzeropulses,nonzerotimes = nonzero_pulses(pulses,times,M=M)

In [ ]:
shots = 100

# 171 s per shot

qtimes = np.zeros(shots,dtype=np.float64)
for j in tqdm(range(shots)):
    pulses = random_pulses_eff(M,K)
    qtime = hamiltonian_engineering(R,K,rows,pulses,beta_HH)[0]
    
    qtimes[j] = qtime
    

  0%|          | 0/100 [00:04<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
mean= np.mean(qtimes)

In [16]:
std = np.std(qtimes)
sem = std/np.sqrt(shots)

In [13]:
mean

np.float64(80.34327141484002)

In [17]:
sem

np.float64(0.1672633884968542)

In [18]:

with h5py.File("quantum_runtimes.hdf5", "w") as slater_file:
    slater_file.create_dataset(
        "qtimes",
        data=qtimes,
        dtype=np.float64,
        chunks=True,
    )